# שבוע 07: ניתוח מלא — מטבעות (מטלה 3)

שיעור זה הוא בסיס המטלה השלישית.  
נבצע את הניתוח המלא: טעינה → GPA → PCA → סטטיסטיקה → פרשנות.

**מטרות:**
- גרף PCA עם אליפסות אמון
- מבחן MANOVA הסתברותי
- השוואת צורות ממוצעות
- ניסוח שאלות מחקר

In [ ]:
!pip install morphops python-bidi -q

import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
import morphops as mops
rtl = get_display
print('הכל מוכן!')

In [ ]:
def parse_tps(filepath_or_text):
    if '\n' in filepath_or_text:
        lines = filepath_or_text.strip().split('\n')
    else:
        with open(filepath_or_text, encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
    specimens, ids = [], []
    i = 0
    while i < len(lines):
        line = lines[i].strip() if hasattr(lines[i], 'strip') else lines[i]
        if line.startswith('LM='):
            n_lm = int(line.split('=')[1])
            coords = []
            for j in range(n_lm):
                i += 1
                parts = lines[i].strip().replace(',', '.').split()
                coords.append([float(parts[0]), float(parts[1])])
            specimens.append(np.array(coords))
        elif line.startswith('ID='):
            ids.append(line.split('=')[1].strip())
        i += 1
    return np.array(specimens), ids

print('parse_tps מוכן')

In [ ]:
import urllib.request

BASE = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/coins/'

def load_tps_url(url):
    with urllib.request.urlopen(url, timeout=15) as r:
        return r.read().decode('utf-8', errors='replace')

try:
    had_text = load_tps_url(BASE + 'hadrian.tps')
    ant_text = load_tps_url(BASE + 'antoninus.tps')
    had_lm, had_ids = parse_tps(had_text)
    ant_lm, ant_ids = parse_tps(ant_text)
    print(f'הדריאנוס: {len(had_lm)} מטבעות, {had_lm.shape[1]} נקודות ציון')
    print(f'אנטונינוס: {len(ant_lm)} מטבעות, {ant_lm.shape[1]} נקודות ציון')
except Exception as e:
    print(f'שגיאה: {e} — משתמשים בנתונים סינתטיים')
    np.random.seed(42)
    n_lm = 16
    angles = np.linspace(0, 2*np.pi, n_lm, endpoint=False)
    base = np.column_stack([np.cos(angles)*100, np.sin(angles)*80])
    had_lm = np.array([base + np.random.randn(n_lm, 2)*5 for _ in range(22)])
    ant_lm = np.array([base*0.9 + np.random.randn(n_lm, 2)*5 + [10,5] for _ in range(15)])
    had_ids = [f'Hadrian_{i+1}' for i in range(22)]
    ant_ids = [f'Antoninus_{i+1}' for i in range(15)]

all_lm = np.concatenate([had_lm, ant_lm], axis=0)
labels = np.array(['Hadrian']*len(had_lm) + ['Antoninus']*len(ant_lm))
print(f'סה"כ: {len(all_lm)} מטבעות')

In [ ]:
result = mops.gpa(all_lm)
aligned = result['aligned']
mean_shape = result['mean']

from sklearn.decomposition import PCA
shape_matrix = aligned.reshape(len(aligned), -1)
pca = PCA()
scores = pca.fit_transform(shape_matrix)
variance_explained = pca.explained_variance_ratio_ * 100

print(f'GPA + PCA הושלמו. PC1: {variance_explained[0]:.1f}%, PC2: {variance_explained[1]:.1f}%')

## גרף PCA עם אליפסות אמון 95%

האליפסות מציגות את אזור ה-95% של כל קבוצה.

In [ ]:
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

def confidence_ellipse(x, y, ax, n_std=2.0, **kwargs):
    if len(x) < 3:
        return
    cov = np.cov(x, y)
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    rx, ry = np.sqrt(1 + pearson), np.sqrt(1 - pearson)
    ellipse = Ellipse((0, 0), width=rx*2, height=ry*2,
                      facecolor='none', **kwargs)
    scale_x = np.sqrt(cov[0, 0]) * n_std
    scale_y = np.sqrt(cov[1, 1]) * n_std
    transf = (transforms.Affine2D()
               .rotate_deg(45)
               .scale(scale_x, scale_y)
               .translate(np.mean(x), np.mean(y)))
    ellipse.set_transform(transf + ax.transData)
    ax.add_patch(ellipse)

fig, ax = plt.subplots(figsize=(9, 7))

mask_h = labels == 'Hadrian'
mask_a = labels == 'Antoninus'

ax.scatter(scores[mask_h, 0], scores[mask_h, 1],
           c='steelblue', s=80, alpha=0.85, label=f'Hadrian (n={mask_h.sum()})', zorder=3)
ax.scatter(scores[mask_a, 0], scores[mask_a, 1],
           c='coral', s=80, marker='s', alpha=0.85,
           label=f'Antoninus (n={mask_a.sum()})', zorder=3)

confidence_ellipse(scores[mask_h, 0], scores[mask_h, 1],
                   ax, edgecolor='steelblue', lw=2, linestyle='--')
confidence_ellipse(scores[mask_a, 0], scores[mask_a, 1],
                   ax, edgecolor='coral', lw=2, linestyle='--')

ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.axvline(0, color='gray', lw=0.8, linestyle=':')
ax.set_xlabel(f'PC1 ({variance_explained[0]:.1f}%)', fontsize=12)
ax.set_ylabel(f'PC2 ({variance_explained[1]:.1f}%)', fontsize=12)
ax.set_title(rtl('מרחב הצורה: מטבעות רומיים — PC1 × PC2'), fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('coins_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('הגרף נשמר כ-coins_pca.png')

In [ ]:
def permutation_manova(X, groups, n_perm=999, seed=42):
    np.random.seed(seed)
    def f_stat(X, g):
        ug = np.unique(g)
        gm = X.mean(axis=0)
        between = sum(np.sum(g==u) * np.sum((X[g==u].mean(0) - gm)**2) for u in ug)
        within  = sum(np.sum((X[g==u] - X[g==u].mean(0))**2) for u in ug)
        return between / within if within > 0 else 0
    obs = f_stat(X, groups)
    perm = [f_stat(X, np.random.permutation(groups)) for _ in range(n_perm)]
    p = (np.sum(np.array(perm) >= obs) + 1) / (n_perm + 1)
    return obs, p, obs / (obs + 1)

print('permutation_manova מוכן')

In [ ]:
# מריצים MANOVA על 4 הרכיבים הראשונים
F_obs, p_val, r_sq = permutation_manova(scores[:, :4], labels, n_perm=999)

print('=== תוצאות MANOVA הסתברותי ===')
print(f'F-statistic: {F_obs:.4f}')
print(f'p-value:     {p_val:.4f}')
print(f'R² (effect): {r_sq:.4f}')
print()
if p_val < 0.05:
    print('** ההפרש בין הקבוצות מובהק סטטיסטית (p < 0.05) **')
else:
    print('אין הפרש מובהק בין הקבוצות (p >= 0.05)')

## צורות ממוצעות לפי קיסר

In [ ]:
had_aligned = aligned[labels == 'Hadrian']
ant_aligned  = aligned[labels == 'Antoninus']
mean_had = had_aligned.mean(axis=0)
mean_ant = ant_aligned.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (title, shape, color) in zip(axes, [
    ('Hadrian', mean_had, 'steelblue'),
    ('Antoninus Pius', mean_ant, 'coral')
]):
    ax.scatter(shape[:, 0], shape[:, 1], color=color, s=80, zorder=3)
    ax.plot(np.append(shape[:, 0], shape[0, 0]),
            np.append(shape[:, 1], shape[0, 1]),
            color=color, lw=1.5)
    # פיזור כל הדגימות ברקע
    group_lm = had_aligned if color == 'steelblue' else ant_aligned
    for specimen in group_lm:
        ax.scatter(specimen[:, 0], specimen[:, 1],
                   color=color, s=8, alpha=0.3)
    ax.set_title(title, fontsize=13)
    ax.set_aspect('equal')
    ax.axis('off')

plt.suptitle(rtl('צורות ממוצעות לפי קיסר (אחרי GPA)'), fontsize=14)
plt.tight_layout()
plt.show()

## טבלת סיכום

| מדד | הדריאנוס | אנטונינוס פיוס |
|-----|----------|----------------|
| מספר מטבעות | 22 | 15 |
| נקודות ציון | 16 | 16 |
| % PC1 | לפי תוצאה | לפי תוצאה |
| p-value MANOVA | לפי תוצאה | — |

## שאלות לדיון (למטלה 3)

1. **הפרדה**: האם שני הקיסרים נפרדים בברור במרחב PCA? מה זה אומר על צורת המטבע?
2. **PC1**: מה מייצג ציר PC1? תיארו בשפה ויזואלית (עגול/אליפסה, שטוח/נפוח).
3. **גודל vs. צורה**: האם גדלי הקנטרואיד שונים? האם ניתן להפריד על סמך גודל בלבד?
4. **פרשנות ארכיאולוגית**: מה ניתן ללמוד על ייצור מטבעות בתקופה הרומית מתוצאות אלה?